# Visual Anagrams — Batch Worker (Free Tier)

This notebook polls Google Drive for queued jobs from the local web app and generates **256×256** rotate-180 visual anagrams on a **free Colab T4** GPU.

**Requirements:** Free Colab (T4 GPU), Hugging Face access to DeepFloyd IF.
Uses less compute than the Pro / 1024×1024 worker — better when you want to stretch monthly Colab units.

The local app uses Google’s `drive.file` scope and **pre-creates** queue/result files
(including a tiny placeholder PNG plus `result_file_id` on each job).
This notebook must **update that same Drive file by ID** via the Drive API — do not
create a new PNG (the local app cannot see Colab-created files under `drive.file`).

**Drive layout used:**
```
My Drive/
├── visual_anagrams/
│   ├── job_queue.json
│   ├── secrets.json
│   └── colab_heartbeat.json
└── visual_anagrams_results/
    └── {job_id}/
        └── image_1024.png   # app placeholder; we overwrite with 256×256 pixels
```

Set runtime to **T4 GPU**, run all cells, then leave the final loop cell running.


## 1. Mount Google Drive


In [ ]:
from google.colab import auth, drive

# Drive API auth so we can overwrite the app-created result PNG by file ID.
auth.authenticate_user()
drive.mount('/content/drive')

# Signal setup progress to the local app (heartbeat file on Drive).
import json
from datetime import datetime, timezone
from pathlib import Path

def _setup_heartbeat(step):
    path = Path('/content/drive/MyDrive/visual_anagrams/colab_heartbeat.json')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({
        'last_seen': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'status': 'starting',
        'host': 'colab',
        'setup_step': step,
    }, indent=2))
    print(f'Setup progress → {step}')

_setup_heartbeat('drive')

## 2. Install dependencies


In [ ]:
# Mark deps as in-progress before the long pip install so the app can show feedback.
import json
from datetime import datetime, timezone
from pathlib import Path

def _setup_heartbeat(step):
    path = Path('/content/drive/MyDrive/visual_anagrams/colab_heartbeat.json')
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({
        'last_seen': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'status': 'starting',
        'host': 'colab',
        'setup_step': step,
    }, indent=2))
    print(f'Setup progress → {step}')

_setup_heartbeat('deps')

!pip install -q -U --no-cache-dir \
  diffusers==0.35.1 \
  transformers==4.55.4 \
  huggingface-hub==0.34.4 \
  safetensors==0.7.0 \
  sentencepiece==0.2.0 \
  accelerate==1.10.1 \
  bitsandbytes==0.49.2 \
  einops==0.7.0 \
  mediapy==1.2.0

!pip install -q --no-cache-dir --no-deps --force-reinstall \
  git+https://github.com/dangeng/visual_anagrams.git

_setup_heartbeat('deps')
print('Dependencies installed')

## 3. Hugging Face login

Reads `visual_anagrams/secrets.json` written by the local app (no need to paste a token here).


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
from huggingface_hub import login

secrets_path = Path('/content/drive/MyDrive/visual_anagrams/secrets.json')
if not secrets_path.exists():
    raise FileNotFoundError(
        f'Missing {secrets_path}. Open the local app, save your Hugging Face token, '
        'and Login with Google so it syncs to Drive.'
    )

secrets = json.loads(secrets_path.read_text())
HF_TOKEN = secrets.get('huggingface_token')
if not HF_TOKEN:
    raise ValueError('secrets.json has no huggingface_token')

login(token=HF_TOKEN)
print('Hugging Face login OK')

hb = Path('/content/drive/MyDrive/visual_anagrams/colab_heartbeat.json')
hb.write_text(json.dumps({
    'last_seen': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'status': 'starting',
    'host': 'colab',
    'setup_step': 'hf',
}, indent=2))
print('Setup progress → hf')


## 4. Paths & helpers


In [ ]:
import json
import os
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

DRIVE_ROOT = Path("/content/drive/MyDrive")
QUEUE_DIR = DRIVE_ROOT / "visual_anagrams"
RESULTS_DIR = DRIVE_ROOT / "visual_anagrams_results"
QUEUE_PATH = QUEUE_DIR / "job_queue.json"
HEARTBEAT_PATH = QUEUE_DIR / "colab_heartbeat.json"
LOCAL_RESULTS = Path("/content/va_results")
MIN_RESULT_BYTES = 10 * 1024

QUEUE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

drive_api = build("drive", "v3", cache_discovery=False)

if not QUEUE_PATH.exists():
    QUEUE_PATH.write_text(json.dumps({"jobs": []}, indent=2))
    print(f"Created empty queue at {QUEUE_PATH}")
else:
    print(f"Queue found at {QUEUE_PATH}")


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def read_queue():
    data = json.loads(QUEUE_PATH.read_text())
    if isinstance(data, list):
        return {"jobs": data}
    data.setdefault("jobs", [])
    return data


def write_queue(queue):
    # Atomic-ish write for Drive
    tmp = QUEUE_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(queue, indent=2))
    tmp.replace(QUEUE_PATH)


def write_heartbeat(extra=None):
    payload = {
        "last_seen": utc_now(),
        "status": "running",
        "host": "colab",
        "setup_step": "ready",
    }
    if extra:
        payload.update(extra)
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))


def update_job(queue, job_id, **fields):
    for job in queue["jobs"]:
        if job["id"] == job_id:
            job.update(fields)
            job["updated_at"] = utc_now()
            return job
    return None


def upload_result_image(file_id, local_path):
    """Overwrite the app-created placeholder by Drive file ID (required for drive.file)."""
    size = local_path.stat().st_size
    if size < MIN_RESULT_BYTES:
        raise RuntimeError(f"Generated image too small ({size} bytes): {local_path}")

    media = MediaFileUpload(str(local_path), mimetype="image/png", resumable=True)
    updated = (
        drive_api.files()
        .update(fileId=file_id, media_body=media, fields="id,size,modifiedTime")
        .execute()
    )
    remote_size = int(updated.get("size") or 0)
    if remote_size < MIN_RESULT_BYTES:
        raise RuntimeError(
            f"Drive update left file too small ({remote_size} bytes) for {file_id}"
        )
    print(f"Uploaded {local_path.name} → file {file_id} ({remote_size} bytes)")
    return updated


write_heartbeat({"status": "starting", "setup_step": "hf"})
print("Helpers ready")

## 5. Clone repo for generate.py (CLI)


In [ ]:
import os
os.chdir("/content")
write_heartbeat({"status": "starting", "setup_step": "clone"})
!rm -rf visual_anagrams_repo
!git clone --depth 1 https://github.com/dangeng/visual_anagrams.git visual_anagrams_repo
REPO = Path("/content/visual_anagrams_repo")
write_heartbeat({"status": "starting", "setup_step": "clone"})
print("Repo ready:", REPO)

## 6. Job runner

Runs `generate.py` at **256×256** (no `--generate_1024`). Honors per-job
`num_inference_steps` from the app, then uploads `sample_256.png` onto the
app-created Drive file (`result_file_id`).


In [ ]:
WORKER_TIER = "free"
DEFAULT_STEPS = 20

def job_wants_1024(job):
    # Free / T4 worker always stays at 256×256.
    return False

def job_steps(job):
    try:
        n = int(job.get("num_inference_steps") or DEFAULT_STEPS)
    except Exception:
        n = DEFAULT_STEPS
    return max(5, min(50, n))

def find_sample(job_id, want_1024=False):
    for name in ["sample_256.png", "sample_1024.png"]:
        direct = LOCAL_RESULTS / job_id / "0000" / name
        if direct.exists():
            return direct
        matches = list((LOCAL_RESULTS / job_id).rglob(name))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No sample_256.png found for {job_id}")

def run_job(job):
    job_id = job["id"]
    prompt_1 = job["prompt_1"]
    prompt_2 = job["prompt_2"]
    seed = job.get("seed", 0)
    result_file_id = job.get("result_file_id")
    if not result_file_id:
        raise RuntimeError(
            f"{job_id} is missing result_file_id. Re-queue from the local app "
            "so it can pre-create the Drive placeholder PNG."
        )

    steps = job_steps(job)

    # Free tier: stop at 256×256 (skips the expensive SD 4× upscaler).
    cmd = [
        "python", "generate.py",
        "--name", job_id,
        "--save_dir", str(LOCAL_RESULTS),
        "--prompts", prompt_1, prompt_2,
        "--views", "identity", "rotate_180",
        "--num_samples", "1",
        "--num_inference_steps", str(steps),
        "--guidance_scale", "10.0",
        "--seed", str(seed),
    ]
    print(f"Running (256, steps={steps}):", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        err = (result.stderr or result.stdout or "").strip() or f"exit {result.returncode}"
        raise RuntimeError(f"generate.py failed:\n{err[-4000:]}")

    src = find_sample(job_id)

    # Update the exact app-owned file — mount copy2 often creates a new invisible file.
    upload_result_image(result_file_id, src)

    # Best-effort mirror onto the mount path for debugging in Drive's UI.
    dest = RESULTS_DIR / job_id / "image_1024.png"
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists():
            with open(dest, "r+b") as f:
                data = src.read_bytes()
                f.truncate(0)
                f.write(data)
                f.flush()
                os.fsync(f.fileno())
        else:
            shutil.copy2(src, dest)
    except Exception as mirror_err:
        print(f"Mount mirror skipped ({mirror_err})")

    return str(src)


## 7. Poll loop (leave this running)

Polls every 30 seconds. Writes a heartbeat (`setup_step` + `last_seen`) so the local app can show online status and setup progress.


In [ ]:
POLL_SECONDS = 30
IDLE_DISCONNECT_MINUTES = 10  # set 0 to disable auto-disconnect
_idle_since = None

print("Batch worker started. Ctrl+C / interrupt to stop.")
write_heartbeat({"message": "worker_started", "setup_step": "ready", "tier": WORKER_TIER})


def maybe_disconnect_if_idle(pending_count):
    """When the queue stays empty, release the Colab GPU so units stop burning."""
    global _idle_since
    if IDLE_DISCONNECT_MINUTES <= 0:
        return
    if pending_count > 0:
        _idle_since = None
        return
    now = time.time()
    if _idle_since is None:
        _idle_since = now
        print(
            f"Queue empty — will auto-disconnect in {IDLE_DISCONNECT_MINUTES} min "
            "to save compute units (Runtime → Disconnect also works anytime)."
        )
        return
    idle_min = (now - _idle_since) / 60.0
    if idle_min < IDLE_DISCONNECT_MINUTES:
        remaining = IDLE_DISCONNECT_MINUTES - idle_min
        print(f"Queue empty — auto-disconnect in {remaining:.1f} min")
        return
    print(
        f"Queue empty for {IDLE_DISCONNECT_MINUTES} min — disconnecting Colab runtime "
        "to stop burning compute units."
    )
    write_heartbeat({
        "status": "offline",
        "setup_step": "ready",
        "tier": WORKER_TIER,
        "message": "auto_disconnected_idle",
    })
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as err:
        print("Auto-disconnect failed:", err)
        print("Please use Runtime → Disconnect and delete runtime manually.")


while True:
    try:
        write_heartbeat({"setup_step": "ready", "tier": WORKER_TIER})
        queue = read_queue()
        pending = [j for j in queue["jobs"] if j.get("status") == "pending"]
        print(f"[{utc_now()}] pending={len(pending)} total={len(queue['jobs'])}")
        maybe_disconnect_if_idle(len(pending))

        for job in pending:
            job_id = job["id"]
            print(f"Processing {job_id}: {job['prompt_1']} ↔ {job['prompt_2']}")
            update_job(queue, job_id, status="processing")
            write_queue(queue)
            write_heartbeat({"current_job": job_id, "setup_step": "ready", "tier": WORKER_TIER})

            try:
                run_job(job)
                update_job(
                    queue,
                    job_id,
                    status="completed",
                    completed_at=utc_now(),
                    error_message=None,
                    image_path="image_1024.png" if WORKER_TIER == "pro" else "sample_256.png",
                    tier=WORKER_TIER,
                )
            except Exception as e:
                print(f"Job {job_id} failed:", e)
                update_job(
                    queue,
                    job_id,
                    status="failed",
                    error_message=str(e),
                )

            write_queue(queue)
            write_heartbeat({
                "last_job": job_id,
                "last_job_status": job.get("status"),
                "setup_step": "ready",
                "tier": WORKER_TIER,
            })

    except Exception as loop_err:
        print("Loop error:", loop_err)
        write_heartbeat({"status": "error", "error": str(loop_err), "setup_step": "ready", "tier": WORKER_TIER})

    time.sleep(POLL_SECONDS)
